# Exclusão de cliente ponta a ponta (etapa 6.4)

Implementa o [ADR 0019](../../docs/adr/0019-exclusao-de-cliente-e-evidencia.md) com o ajuste do [ADR 0020](../../docs/adr/0020-propagacao-da-exclusao-por-reconstrucao-e-purga-fisica.md): a pessoa (`customer_unique_id`) é apagada da **bronze**, as tabelas derivadas da silver são **reconstruídas** (full refresh seletivo) e os arquivos antigos são **purgados**. O pedido permanece, sem vínculo com a pessoa.

**Fluxo do modo `executar`**

1. registra a solicitação em `ops.exclusoes_lgpd` (só o **hash** do titular);
2. `DELETE` na `bronze.customers` (por tabela de alvos, para o id não aparecer no histórico da tabela);
3. `REORG ... APPLY (PURGE)` e `VACUUM` na bronze;
4. full refresh seletivo no pipeline `olist.silver` (`customers`, `customers_historico`, `customers_quarentena`) e atualização do pipeline `olist.gold`;
5. `REORG` e `VACUUM` nas tabelas derivadas e nas cópias físicas das materialized views;
6. verificação independente e fechamento da solicitação.

**Parâmetros**

| Widget | Para que serve |
|---|---|
| `modo` | `simulacao` (padrão: só conta e mostra) ou `executar` (apaga de verdade; **irreversível**) |
| `customer_unique_id` | identificador da pessoa a excluir |
| `confirmacao` | no modo `executar`, digite `EXCLUIR` seguido do código de 8 caracteres que a simulação mostra |

> **Cuidados:** rode como `olist_engenharia` (o filtro de linha e a máscara não podem esconder linhas do procedimento). O notebook nunca imprime o identificador; só o hash curto. Os arquivos brutos da landing (`raw_data`) ficam fora do escopo (ADR 0020).

In [ ]:
# O QUE FAZ: define parâmetros e constantes do procedimento.
#
# FUNÇÕES DECLARADAS:
# - nenhuma (configuração).
#
# FUNÇÕES NATIVAS E SQL:
# - dbutils.widgets.dropdown / text / get: parâmetros no topo do notebook.
# - str.strip(): tira espaços do identificador digitado.
#
# REGRAS:
# - O identificador nunca é impresso; o notebook mostra só o hash curto (8 caracteres).
# - TABELAS_BASE: tabelas que guardam a pessoa (bronze, silver, quarentena e gold); cada uma tem uma cópia física `__materialization_*`
#   quando é gerida pelo pipeline, e essa cópia também é limpa.
from pyspark.sql import functions as F

CATALOG = "olist"
dbutils.widgets.dropdown("modo", "simulacao", ["simulacao", "executar"], "Modo")
dbutils.widgets.text("customer_unique_id", "", "customer_unique_id do titular")
dbutils.widgets.text("confirmacao", "", "Confirmação (só no executar)")
MODO = dbutils.widgets.get("modo")
TITULAR = dbutils.widgets.get("customer_unique_id").strip()
CONFIRMACAO = dbutils.widgets.get("confirmacao").strip()
assert TITULAR, "Informe o customer_unique_id no widget."

TABELAS_BASE = ["bronze.customers", "silver.customers", "silver.customers_historico",
                "quality.customers_quarentena", "gold.dim_customer", "gold.fact_orders"]
FULL_REFRESH_SILVER = [f"{CATALOG}.silver.customers", f"{CATALOG}.silver.customers_historico", f"{CATALOG}.quality.customers_quarentena"]
print("modo:", MODO)

In [ ]:
# O QUE FAZ: prepara a tabela de verificações (a mesma da 6.2), a tabela de solicitações e a de alvos, e calcula o hash do titular.
#
# FUNÇÕES DECLARADAS:
# - registrar(etapa, linhas, apagar_antes): apaga as linhas anteriores da etapa (ou dos perfis) e grava as novas em
#   quality.validacao_governanca. Rodar de novo troca o resultado, não duplica.
#
# FUNÇÕES NATIVAS E SQL:
# - CREATE TABLE IF NOT EXISTS ... COMMENT: cria a tabela de resultados uma vez, já com comentário em tabela e colunas.
# - DELETE FROM t WHERE ...: remove as linhas antigas (Delta). spark.createDataFrame(linhas, "schema DDL"): cria o
#   DataFrame a partir de tuplas. F.lit(x): coluna com valor fixo. F.current_timestamp(): hora da execução.
# - df.write.mode("append").saveAsTable(t): acrescenta as linhas à tabela.
# - hash_titular(valor): sha2(concat(salt, valor), 256), o mesmo hash das máscaras; só ele é gravado.
#
# FUNÇÕES NATIVAS E SQL:
# - CREATE TABLE IF NOT EXISTS ... COMMENT: ops.exclusoes_lgpd (solicitações) e ops.exclusoes_alvos (ids do titular durante o procedimento).
# - sha2(concat(salt, valor), 256): hash irreversível com o salt guardado em ops.governanca_config.

VALIDACAO = f"{CATALOG}.quality.validacao_governanca"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {VALIDACAO} (
  etapa STRING COMMENT 'Etapa da Fase 6 que gerou a verificação (6.2-grants, 6.2-mascaras, 6.2-perfil, ...).',
  perfil STRING COMMENT 'Perfil avaliado (analista, gestor, dpo, engenharia) ou geral quando vale para todos.',
  verificacao STRING COMMENT 'O que foi verificado.',
  esperado STRING COMMENT 'Valor esperado pela regra.',
  obtido STRING COMMENT 'Valor encontrado no ambiente.',
  resultado STRING COMMENT 'OK, REPROVADA ou REGISTRO (só guarda um valor de referência).',
  executado_em TIMESTAMP COMMENT 'Momento da execução.'
)
COMMENT 'Resultado das verificações de governança e LGPD (grants, máscaras, filtros de linha, exclusão de cliente). Nenhuma linha guarda dado pessoal.'
""")


def registrar(etapa: str, linhas: list, apagar_antes: str = "etapa") -> None:
    """Grava as verificações. linhas = [(perfil, verificacao, esperado, obtido, resultado)]."""
    if apagar_antes == "etapa":
        spark.sql(f"DELETE FROM {VALIDACAO} WHERE etapa = '{etapa}'")
    else:
        perfis = sorted({l[0] for l in linhas})
        lista = ", ".join(f"'{p}'" for p in perfis)
        spark.sql(f"DELETE FROM {VALIDACAO} WHERE etapa = '{etapa}' AND perfil IN ({lista})")
    df = (spark.createDataFrame(linhas, "perfil string, verificacao string, esperado string, obtido string, resultado string")
          .withColumn("etapa", F.lit(etapa))
          .withColumn("executado_em", F.current_timestamp())
          .select("etapa", "perfil", "verificacao", "esperado", "obtido", "resultado", "executado_em"))
    df.write.mode("append").saveAsTable(VALIDACAO)


spark.sql(f"""CREATE TABLE IF NOT EXISTS {CATALOG}.ops.exclusoes_lgpd (
  solicitacao_id STRING COMMENT 'Identificador da solicitação de exclusão.',
  titular_hash STRING COMMENT 'Hash com salt do customer_unique_id; o dado em claro nunca é gravado.',
  solicitada_em TIMESTAMP COMMENT 'Início do procedimento.',
  concluida_em TIMESTAMP COMMENT 'Fim do procedimento (nulo enquanto não termina).',
  status STRING COMMENT 'iniciada, concluida, sem_dados (titular não encontrado) ou falhou.',
  qtd_customer_ids BIGINT COMMENT 'Quantos customer_id a pessoa tinha.',
  qtd_pedidos BIGINT COMMENT 'Quantos pedidos permanecem sem vínculo com a pessoa.',
  contagens_antes STRING COMMENT 'Linhas do titular por tabela antes da exclusão (JSON).',
  contagens_depois STRING COMMENT 'Linhas do titular por tabela depois da exclusão (JSON); deve ser tudo 0.',
  versoes_delta STRING COMMENT 'Versão Delta de cada tabela antes e depois (JSON).')
  COMMENT 'Registro das solicitações de exclusão de cliente (LGPD). Guarda o hash do titular e contagens, nunca o dado pessoal em claro.'""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {CATALOG}.ops.exclusoes_alvos (
  customer_unique_id STRING COMMENT 'Titular em exclusão; a linha é apagada ao fim do procedimento.',
  customer_id STRING COMMENT 'customer_id do titular; a linha é apagada ao fim do procedimento.')
  COMMENT 'Tabela transitória: ids do titular usados nos DELETE, para o valor não aparecer no histórico das tabelas. Fica vazia fora do procedimento.'""")


def hash_titular(valor: str) -> str:
    salt = spark.sql(f"SELECT valor FROM {CATALOG}.ops.governanca_config WHERE chave = 'salt_pii'").first()[0]
    return spark.sql("SELECT sha2(concat(:s, :v), 256)", args={"s": salt, "v": valor}).first()[0]


HASH = hash_titular(TITULAR)
CODIGO = HASH[:8]
print("titular (hash curto):", CODIGO)

In [ ]:
# O QUE FAZ: localiza o titular, conta as linhas por tabela e tira a foto dos números oficiais antes da exclusão.
#
# FUNÇÕES DECLARADAS:
# - contar(tabela, ids): linhas do titular na tabela, por customer_unique_id, customer_id ou linha bruta (_corrupt_record).
# - indicadores(): totais da gold (pedidos, válidos, GMV, entregues, no prazo, clientes e taxa de recompra) para comparar antes e depois.
#
# FUNÇÕES NATIVAS E SQL:
# - spark.table(...).where(...).count(): conta linhas; collect() só nos poucos customer_id do titular.
# - F.col(...).contains(texto): procura o identificador dentro da linha bruta.
# - F.countDistinct, F.sum, F.when: indicadores agregados da fact_orders.
def colunas(tabela: str) -> set:
    return set(spark.table(f"{CATALOG}.{tabela}").columns)


def condicao(tabela: str, ids: list):  # fact_orders conta só pelo customer_unique_id (o pedido continua existindo)
    cols = colunas(tabela)
    c = F.lit(False)
    if "customer_unique_id" in cols:
        c = c | (F.col("customer_unique_id") == TITULAR)
    if "customer_id" in cols and ids and tabela != "gold.fact_orders":  # o pedido permanece com o customer_id
        c = c | F.col("customer_id").isin(ids)
    if "_corrupt_record" in cols:
        c = c | F.col("_corrupt_record").contains(TITULAR)
    return c


def contar(tabela: str, ids: list) -> int:
    return spark.table(f"{CATALOG}.{tabela}").where(condicao(tabela, ids)).count()


def indicadores() -> dict:
    fo = spark.table(f"{CATALOG}.gold.fact_orders")
    base = fo.agg(
        F.count("*").alias("pedidos"),
        F.sum(F.col("pedido_valido").cast("int")).alias("pedidos_validos"),
        F.sum(F.when(F.col("pedido_valido"), F.col("gmv_bruto")).otherwise(0)).alias("gmv_valido"),
        F.sum(F.col("entregue").cast("int")).alias("entregues"),
        F.sum(F.col("entregue_no_prazo").cast("int")).alias("entregues_no_prazo"),
    ).first().asDict()
    pessoas = (fo.where(F.col("pedido_valido") & F.col("customer_unique_id").isNotNull())
                 .groupBy("customer_unique_id").count())
    base["clientes_com_pedido"] = pessoas.count()
    base["clientes_recompra"] = pessoas.where("count >= 2").count()
    return {k: str(v) for k, v in base.items()}


ids = [r[0] for r in (spark.table(f"{CATALOG}.bronze.customers").where(F.col("customer_unique_id") == TITULAR)
                      .select("customer_id").union(spark.table(f"{CATALOG}.silver.customers_historico")
                      .where(F.col("customer_unique_id") == TITULAR).select("customer_id")).distinct().collect())]
antes = {t: contar(t, ids) for t in TABELAS_BASE}
pedidos = spark.table(f"{CATALOG}.gold.fact_orders").where(F.col("customer_id").isin(ids)).count() if ids else 0
ind_antes = indicadores()

print(f"titular {CODIGO}: {len(ids)} customer_id, {pedidos} pedido(s) permanecem sem vínculo")
for t, q in antes.items():
    print(f"  {t:32} {q}")
print("indicadores antes:", ind_antes)
if not any(antes.values()):
    print("\nTitular não encontrado em nenhuma tabela: nada a excluir (a solicitação fica como sem_dados no modo executar).")
print(f"\nPara excluir de verdade: modo = executar e confirmacao = EXCLUIR{CODIGO}")

In [ ]:
# O QUE FAZ: modo `executar`, parte 1. Registra a solicitação, apaga o titular da bronze e purga os arquivos antigos da bronze.
#
# FUNÇÕES DECLARADAS:
# - versao(tabela): versão Delta atual da tabela (DESCRIBE HISTORY), para a evidência antes e depois.
# - purgar(tabela): REORG ... APPLY (PURGE) e VACUUM com retenção zero, só dentro do procedimento.
#
# FUNÇÕES NATIVAS E SQL:
# - INSERT INTO ops.exclusoes_alvos: guarda os ids; o DELETE usa subconsulta nela, e o histórico da tabela mostra a subconsulta, não o id.
# - DELETE FROM ... WHERE ... IN (SELECT ...) / EXISTS: três comandos separados (por identificador, por customer_id e pela linha bruta); apaga as linhas do titular (marca nos deletion vectors).
# - REORG TABLE ... APPLY (PURGE): reescreve os arquivos sem as linhas marcadas.
# - ALTER TABLE ... SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours'): baixa o mínimo de retenção da tabela
#   (o serverless não deixa desligar a verificação de segurança do VACUUM); a propriedade é removida logo depois.
# - VACUUM tabela RETAIN 0 HOURS: apaga os arquivos antigos (e os do Change Data Feed) que ainda têm a pessoa.
import json, uuid

SOLICITACAO = str(uuid.uuid4())
FALHAS = []


def versao(tabela: str) -> int:
    try:
        return int(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{tabela} LIMIT 1").first()["version"])
    except Exception:
        return -1  # materialized view: não tem histórico Delta próprio (a cópia física `__materialization_*` tem)


def purgar(tabela: str) -> None:
    alvo = f"{CATALOG}.{tabela}"
    for etapa, sql in [("reorg", f"REORG TABLE {alvo} APPLY (PURGE)"),
                       ("retencao", f"ALTER TABLE {alvo} SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')"),
                       ("vacuum", f"VACUUM {alvo} RETAIN 0 HOURS"),
                       ("restaura", f"ALTER TABLE {alvo} UNSET TBLPROPERTIES ('delta.deletedFileRetentionDuration')")]:
        try:
            spark.sql(sql)
        except Exception as e:
            FALHAS.append((f"{etapa} {tabela}", str(e)[:200]))
            print("FALHA", etapa, tabela, "->", str(e)[:140])
    print("ok   ", tabela)


RETOMAR = False
if MODO == "executar":
    aberta = (spark.table(f"{CATALOG}.ops.exclusoes_lgpd").where((F.col("titular_hash") == HASH) & F.col("status").isin("iniciada", "falhou"))
              .orderBy(F.col("solicitada_em").desc()).first())
    if not any(antes.values()) and aberta is None:
        spark.sql(f"""INSERT INTO {CATALOG}.ops.exclusoes_lgpd VALUES ('{SOLICITACAO}', '{HASH}', current_timestamp(),
                      current_timestamp(), 'sem_dados', 0, 0, '{json.dumps(antes)}', '{json.dumps(antes)}', '{{}}')""")
        raise SystemExit("Titular não encontrado: solicitação registrada como sem_dados. Nada foi apagado.")
    assert CONFIRMACAO == f"EXCLUIR{CODIGO}", f"Confirmação incorreta. Digite EXCLUIR{CODIGO} (o código aparece na simulação)."

    if not any(antes.values()):
        # Retomada: o titular já saiu de todas as tabelas, mas a solicitação anterior não terminou (purga ou verificação pendente).
        RETOMAR = True
        SOLICITACAO = aberta["solicitacao_id"]
        antes = json.loads(aberta["contagens_antes"], strict=False)
        v_antes = json.loads(aberta["versoes_delta"], strict=False)["antes"] if aberta["versoes_delta"] else {}
        print("Retomando a solicitação", SOLICITACAO[:8], "(purga e verificação)")
    else:
        v_antes = {t: versao(t) for t in TABELAS_BASE}
        spark.sql(f"""INSERT INTO {CATALOG}.ops.exclusoes_lgpd VALUES ('{SOLICITACAO}', '{HASH}', current_timestamp(), NULL,
                      'iniciada', {len(ids)}, {pedidos}, '{json.dumps(antes)}', NULL, '{json.dumps({"antes": v_antes})}')""")
        spark.sql(f"DELETE FROM {CATALOG}.ops.exclusoes_alvos")
        linhas = [(TITULAR, i) for i in ids]
        spark.createDataFrame(linhas, "customer_unique_id string, customer_id string").write.mode("append").saveAsTable(f"{CATALOG}.ops.exclusoes_alvos")

        alvos = f"{CATALOG}.ops.exclusoes_alvos"
        b = f"{CATALOG}.bronze.customers"
        spark.sql(f"DELETE FROM {b} WHERE customer_unique_id IN (SELECT customer_unique_id FROM {alvos})")
        spark.sql(f"DELETE FROM {b} WHERE customer_id IN (SELECT customer_id FROM {alvos})")
        spark.sql(f"DELETE FROM {b} WHERE EXISTS (SELECT 1 FROM {alvos} a WHERE _corrupt_record LIKE concat('%', a.customer_unique_id, '%'))")
        print("ok    DELETE em bronze.customers")
    purgar("bronze.customers")
else:
    print("Modo simulacao: nada foi apagado.")

In [ ]:
# O QUE FAZ: modo `executar`, parte 2. Reconstrói as tabelas derivadas da silver e atualiza a gold.
#
# FUNÇÕES DECLARADAS:
# - esperar(pipeline_id, update_id): consulta o estado da atualização até ela terminar; devolve o estado final.
# - pipeline_id(nome): procura o pipeline pelo nome.
#
# FUNÇÕES NATIVAS E SQL:
# - WorkspaceClient().pipelines.start_update(pipeline_id, full_refresh_selection=[...]): full refresh só das tabelas listadas
#   (reprocessa a partir da bronze já sem a pessoa).
# - WorkspaceClient().pipelines.start_update(pipeline_id): atualização normal da gold (as materialized views refletem a silver).
# - WorkspaceClient().pipelines.get_update: estado da atualização (QUEUED, RUNNING, COMPLETED, FAILED, CANCELED).
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()


def pipeline_id(nome: str) -> str:
    return next(p.pipeline_id for p in w.pipelines.list_pipelines(filter=f"name LIKE '{nome}'"))


def esperar(pid: str, uid: str) -> str:
    while True:
        estado = w.pipelines.get_update(pid, uid).update.state.value
        if estado in ("COMPLETED", "FAILED", "CANCELED"):
            return estado
        time.sleep(20)


if MODO == "executar" and not RETOMAR:
    pid_silver = pipeline_id("olist.silver")
    u = w.pipelines.start_update(pid_silver, full_refresh_selection=FULL_REFRESH_SILVER).update_id
    print("silver (full refresh seletivo):", esperar(pid_silver, u))
    pid_gold = pipeline_id("olist.gold")
    u = w.pipelines.start_update(pid_gold).update_id
    estado = esperar(pid_gold, u)
    print("gold (atualização):", estado)
    assert estado == "COMPLETED", "A atualização da gold não terminou; veja o pipeline antes de continuar."

In [ ]:
# O QUE FAZ: modo `executar`, parte 3. Purga os arquivos antigos das tabelas derivadas e das cópias físicas das materialized views.
#
# FUNÇÕES DECLARADAS:
# - copias(tabela): nomes das tabelas `__materialization_*` ligadas à tabela (cópia física gerida pelo pipeline).
#
# FUNÇÕES NATIVAS E SQL:
# - information_schema.tables LIKE '%__materialization_%_<nome>_%': acha a cópia física de cada materialized view ou tabela do pipeline.
# - purgar(tabela): REORG, VACUUM com retenção zero e restauração da propriedade (definida na parte 1).
def copias(tabela: str) -> list:
    esquema, nome = tabela.split(".")
    achadas = spark.sql(f"""SELECT table_name FROM {CATALOG}.information_schema.tables
                            WHERE table_schema = '{esquema}' AND table_name LIKE '\\_\\_materialization\\_%\\_{nome}\\_%'""").collect()
    return [f"{esquema}.{r[0]}" for r in achadas]


LIMPEZA = []
if MODO == "executar":
    for t in TABELAS_BASE:
        if t == "bronze.customers":
            continue
        tipo = spark.sql(f"SELECT table_type FROM {CATALOG}.information_schema.tables "
                         f"WHERE table_schema = '{t.split('.')[0]}' AND table_name = '{t.split('.')[1]}'").first()[0]
        LIMPEZA += ([] if tipo in ("MATERIALIZED_VIEW", "STREAMING_TABLE") else [t]) + copias(t)  # tabela do pipeline: os arquivos ficam na cópia física
    for t in LIMPEZA:
        purgar(t)
    print(f"\n{len(LIMPEZA)} tabelas purgadas; falhas: {len(FALHAS)}")

In [ ]:
# O QUE FAZ: modo `executar`, parte 4. Verificação independente, evidência e fechamento da solicitação.
#
# FUNÇÕES DECLARADAS:
# - nenhuma (usa contar, indicadores, versao e registrar).
#
# FUNÇÕES NATIVAS E SQL:
# - DESCRIBE HISTORY: lista as operações da tabela; procura-se o identificador nos parâmetros (não pode aparecer).
# - SELECT ... VERSION AS OF n: leitura em versão antiga (time travel); depois do VACUUM deve falhar ou não trazer a pessoa.
# - DELETE FROM ops.exclusoes_alvos + VACUUM: esvazia a tabela transitória de ids.
# - UPDATE ops.exclusoes_lgpd: fecha a solicitação com as contagens e as versões.
if MODO == "executar":
    depois = {t: contar(t, ids) for t in TABELAS_BASE}
    ind_depois = indicadores()
    linhas = [("geral", f"titular com 0 linhas em {t}", "0", str(q), "OK" if q == 0 else "REPROVADA") for t, q in depois.items()]

    iguais = ["pedidos", "pedidos_validos", "gmv_valido", "entregues", "entregues_no_prazo"]
    for k in iguais:
        linhas.append(("geral", f"indicador inalterado: {k}", ind_antes[k], ind_depois[k], "OK" if ind_antes[k] == ind_depois[k] else "REPROVADA"))
    # A pessoa sai do cálculo de recompra (ADR 0020): clientes com pedido cai em 1 por pessoa que tinha pedido válido; registra a diferença.
    linhas.append(("geral", "clientes com pedido válido (diferença esperada: a pessoa saiu)", ind_antes["clientes_com_pedido"],
                   ind_depois["clientes_com_pedido"], "REGISTRO"))
    linhas.append(("geral", "clientes com 2+ pedidos (recompra)", ind_antes["clientes_recompra"], ind_depois["clientes_recompra"], "REGISTRO"))

    vazamento = 0
    for t in TABELAS_BASE:
        for tt in [t] + copias(t):
            try:
                hist = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{tt}").select("operationParameters").collect()
            except Exception:
                continue  # materialized view: só a cópia física tem histórico
            vazamento += sum(1 for r in hist if TITULAR in str(r[0]))
    linhas.append(("geral", "identificador ausente no histórico das tabelas (DESCRIBE HISTORY)", "0", str(vazamento), "OK" if vazamento == 0 else "REPROVADA"))

    try:
        assert "bronze.customers" in v_antes, "sem versão anterior registrada"
        n_t = spark.sql(f"SELECT count(*) FROM {CATALOG}.bronze.customers VERSION AS OF {v_antes['bronze.customers']} "
                        f"WHERE customer_unique_id = '{TITULAR}'").first()[0]
        resultado, ok_tt = f"{n_t} linhas do titular na versão antiga", n_t == 0
    except AssertionError:
        resultado, ok_tt = "versão anterior não registrada (retomada)", True
    except Exception:
        resultado, ok_tt = "leitura da versão antiga bloqueada pela purga", True
    linhas.append(("geral", "time travel não devolve a pessoa", "0 linhas do titular", resultado, "OK" if ok_tt else "REPROVADA"))
    linhas.append(("geral", "comandos de purga sem falha", "0", str(len(FALHAS)), "OK" if not FALHAS else "REPROVADA"))

    registrar("6.4-exclusao", linhas, apagar_antes="perfil")
    for l in linhas:
        print(f"[{l[4]:9}] {l[1]} | esperado: {l[2]} | obtido: {l[3]}")

    v_depois = {t: versao(t) for t in TABELAS_BASE}
    spark.sql(f"DELETE FROM {CATALOG}.ops.exclusoes_alvos")
    purgar("ops.exclusoes_alvos")
    NL, ASPAS = chr(10), chr(39)
    falhas_txt = [(f[0] + ": " + f[1]).replace(NL, " ").replace(ASPAS, "") for f in FALHAS]
    spark.sql(f"""UPDATE {CATALOG}.ops.exclusoes_lgpd
                  SET status = '{"concluida" if all(l[4] != "REPROVADA" for l in linhas) else "falhou"}', concluida_em = current_timestamp(),
                      contagens_depois = '{json.dumps(depois)}',
                      versoes_delta = '{json.dumps({"antes": v_antes, "depois": v_depois, "falhas": falhas_txt})}'
                  WHERE solicitacao_id = '{SOLICITACAO}'""")
    spark.sql(f"""UPDATE {CATALOG}.ops.exclusoes_lgpd SET status = 'interrompida', concluida_em = current_timestamp()
                  WHERE titular_hash = '{HASH}' AND status = 'iniciada' AND solicitacao_id <> '{SOLICITACAO}'""")
    if any(l[4] == "REPROVADA" for l in linhas):
        raise RuntimeError("Verificação reprovada; veja a tabela quality.validacao_governanca e ops.exclusoes_lgpd")
    print("\nExclusão concluída. Evidência em ops.exclusoes_lgpd e quality.validacao_governanca (etapa 6.4-exclusao).")
else:
    print("Rode com modo = executar para excluir.")